In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
import timm
from PIL import Image

# ==========================================
# 1. CONFIGURAZIONE AMBIENTE E PATH (Drive)
# ==========================================
# In Google Colab, assicurati di eseguire prima:
# from google.colab import drive
# drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Progetto_EVWSD_ML'
CACHE_DIR = os.path.join(PROJECT_DIR, 'data/hf_cache')
EMBEDDINGS_DIR = os.path.join(PROJECT_DIR, 'embeddings')

# Creiamo le cartelle di lavoro se non esistono
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)

# Impostiamo il dispositivo (GPU se disponibile, altrimenti CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo di calcolo attivo: {device}")

# ==========================================
# 2. DEFINIZIONE DEL DATASET CUSTOM (PyTorch)
# ==========================================
class EVWSDImageDataset(Dataset):
    """
    Dataset personalizzato per gestire le immagini del task EVWSD-ITA.
    Prende i dati direttamente dall'oggetto Dataset di Hugging Face.
    """
    def __init__(self, hf_dataset, split="train"):
        # Selezioniamo lo split corretto (train, validation o test)
        self.data = hf_dataset[split]
        
        # Pipeline di trasformazione (estratta dal Lab 04)
        # Ridimensiona, converte in tensore e applica la normalizzazione standard di ImageNet
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Recuperiamo la riga corrente del dataset
        item = self.data[idx]
        
        # Nel dataset EVWSD-ITA ci sono solitamente 10 immagini candidate per riga.
        # Estraiamo i campi immagine (es. image_1, image_2 ... image_10) dinamitamente.
        candidate_images = []
        for i in range(1, 11):
            img_key = f'image_{i}'
            if img_key in item:
                img = item[img_key]
                # Assicuriamoci che l'immagine sia in formato RGB (evita errori con PNG a 4 canali o scala di grigi)
                if isinstance(img, Image.Image):
                    img = img.convert("RGB")
                    candidate_images.append(self.transform(img))
                
        # Impiliamo i tensori delle 10 immagini candidate
        # Forma risultante: [10, 3, 224, 224] (10 immagini, 3 canali, altezza, larghezza)
        stacked_images = torch.stack(candidate_images)
        
        # Recuperiamo anche l'ID o la query per tenere traccia dell'associazione
        query_id = item.get('id', idx)
        
        return stacked_images, query_id

# ==========================================
# 3. CARICAMENTO MODELLO PRE-ADDESTRATO
# ==========================================
def get_feature_extractor(model_name='vit_base_patch16_224'):
    """
    Carica un modello pre-addestrato dalla libreria 'timm' (visto nel Lab 04)
    escludendo la testa di classificazione finale per usarlo come estrattore di feature.
    """
    print(f"Caricamento del modello {model_name}...")
    # Impostando num_classes=0 indichiamo a timm di rimuovere lo strato di classificazione lineare finale
    model = timm.create_model(model_name, pretrained=True, num_classes=0)
    model = model.to(device)
    # Impostiamo il modello in modalità valutazione (visto nel Lab 02)
    model.eval()
    return model

# ==========================================
# 4. PIPELINE DI ESTRAZIONE E SALVATAGGIO
# ==========================================
def extract_and_save_embeddings(dataloader, model, output_filename="tensor_immagini.pt"):
    """
    Scorre l'intero dataloader, estrae le feature visive per ogni immagine candidata
    e salva il mega-tensore risultante su Google Drive.
    """
    all_embeddings = []
    all_ids = []
    
    # Disattiviamo il calcolo dei gradienti per risparmiare memoria (visto nel Lab 02)
    with torch.no_grad():
        for batch_idx, (images_batch, ids) in enumerate(dataloader):
            # images_batch ha dimensione: [BatchSize, 10, 3, 224, 224]
            batch_size = images_batch.size(0)
            
            # Per far passare le immagini nel modello, dobbiamo appiattire la dimensione del batch
            # Convertiamo temporaneamente da [B, 10, 3, 224, 224] a [B*10, 3, 224, 224]
            flat_images = images_batch.view(-1, 3, 224, 224).to(device)
            
            # Estraiamo le feature (es. un vettore di dimensione 768 per ogni immagine usando ViT)
            features = model(flat_images) 
            
            # Ripristiniamo la struttura originale dividendo nuovamente per le 10 immagini candidate
            # Forma finale per questo batch: [BatchSize, 10, feature_dim]
            features_dim = features.size(-1)
            reshaped_features = features.view(batch_size, 10, features_dim)
            
            # Spostiamo i vettori sulla CPU per non saturare la memoria della GPU
            all_embeddings.append(reshaped_features.cpu())
            all_ids.extend(ids if isinstance(ids, list) else ids.tolist())
            
            if (batch_idx + 1) % 10 == 0:
                print(f"Processati {batch_idx + 1}/{len(dataloader)} batch...")
                
    # Uniamo tutti i batch in un unico tensore globale
    final_embeddings = torch.cat(all_embeddings, dim=0)
    
    # Salviamo il dizionario contenente vettori e ID associati su Google Drive
    save_path = os.path.join(EMBEDDINGS_DIR, output_filename)
    torch.save({
        'embeddings': final_embeddings,
        'ids': all_ids
    }, save_path)
    
    print(f"\nEstrazione completata con successo!")
    print(f"Tensore salvato in: {save_path}")
    print(f"Forma finale del tensore degli embedding: {final_embeddings.shape}")

# ==========================================
# 5. CODICE DI TEST PRINCIPALE
# ==========================================
if __name__ == "__main__":
    # 1. Carichiamo il dataset ufficiale EVWSD-ITA tramite Hugging Face
    print("Inizializzazione del dataset da Hugging Face...")
    hf_dataset = load_dataset("swap-uniba/EVWSD-ITA", cache_dir=CACHE_DIR)
    
    # 2. Creiamo l'istanza del nostro dataset PyTorch (lato Immagini)
    train_dataset = EVWSDImageDataset(hf_dataset, split="train")
    
    # 3. Creiamo il DataLoader di PyTorch per caricare i dati a blocchi (batch)
    # Batch size = 8 o 16 è ideale per non terminare la memoria RAM di Colab
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=False, num_workers=2)
    
    # 4. Carichiamo l'estrattore di feature (Vision Transformer)
    # Nota: Puoi sostituirlo con 'resnet50' o altri modelli supportati da timm
    feature_extractor = get_feature_extractor('vit_base_patch16_224')
    
    # 5. Avviamo l'estrazione e salviamo il file su Drive
    print("Avvio della pipeline di estrazione degli embedding...")
    extract_and_save_embeddings(train_loader, feature_extractor, output_filename="embeddings_immagini_train.pt")